# 切块和索引

上一章的语义搜索能按意思找到相关的句子，但离真正用起来还差两步：

1. **资料是手写的一句一句话**。真正的资料是一篇篇文章，甚至一整本书。直接把一整篇文章变成一个向量，搜出来的结果太粗：小明只想知道一个小问题，却拿到一整章的内容。
2. **每次搜索都要重新计算所有资料的向量**。资料一多，又慢又费钱。可资料本身并没有变，向量完全可以算一次，存起来反复用。

这一章就来解决这两个问题：

- **切块**（chunking）：把文章切成一小段一小段；
- **索引**（index）：提前算好每一段的向量，保存到文件里，以后直接加载、直接查。

就像一本书的目录：编一次，以后每次查找都很快。

导入的部分多了 `re`（正则表达式）。

In [ ]:
import asyncio
import inspect
import json
import os
import re
import threading
import uuid
from abc import ABC, abstractmethod
from datetime import datetime, timedelta
from enum import Enum
from pathlib import Path
from typing import Annotated, Callable, List, get_args, get_origin, get_type_hints
from zoneinfo import ZoneInfo

import anthropic
import numpy as np
from anthropic.types import UserMessageParam, AssistantMessageParam, ToolParam, TextBlockParam, ToolUseBlockParam, ToolResultBlockParam
from mcp import Client
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam
from openai.types.chat import ChatCompletionToolParam, ChatCompletionMessageToolCallParam, ChatCompletionToolMessageParam

In [ ]:
def note(label: str, text: str):
    """Print one step of the agent's work, indented, so it stands apart from questions and answers."""
    lines = str(text).splitlines() or [""]
    print(f"  * {label}: {lines[0]}")
    for line in lines[1:]:
        print(f"    {line}")

In [ ]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"
    TOOL = "Tool"

In [ ]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.summary()}

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [ ]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [ ]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [ ]:
class ToolCall:

    def __init__(self, id: str, name: str, arguments: dict):
        self.id = id
        self.name = name
        self.arguments = arguments

    def to_dict(self) -> dict:
        return {"id": self.id, "name": self.name, "arguments": self.arguments}

    def __repr__(self) -> str:
        return f"{self.name}({json.dumps(self.arguments, ensure_ascii=False)})"

In [ ]:
class AssistantMessage(Message):

    def __init__(self, content: str, tool_calls: List[ToolCall] = None):
        super().__init__(MsgType.ASSISTANT)
        self.content = content
        self.tool_calls = tool_calls or []

    def summary(self) -> str:
        if self.tool_calls:
            return f"{self.content} [call: {self.tool_calls}]"
        return self.content

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.content,
                "tool_calls": [call.to_dict() for call in self.tool_calls]}

In [ ]:
class ToolResultMessage(Message):

    def __init__(self, tool_call_id: str, name: str, result: str):
        super().__init__(MsgType.TOOL)
        self.tool_call_id = tool_call_id
        self.name = name
        self.result = result

    def summary(self) -> str:
        return f"{self.name} -> {self.result}"

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "tool_call_id": self.tool_call_id,
                "name": self.name, "content": self.result}

In [ ]:
def message_from_dict(data: dict) -> Message:
    if data["type"] == MsgType.ASSISTANT.value:
        return AssistantMessage(data["content"], [ToolCall(**c) for c in data.get("tool_calls", [])])
    if data["type"] == MsgType.TOOL.value:
        return ToolResultMessage(data["tool_call_id"], data["name"], data["content"])
    classes = {MsgType.SYSTEM.value: SystemMessage, MsgType.USER.value: UserMessage}
    return classes[data["type"]](data["content"])

In [ ]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

In [ ]:
class Tool:

    def __init__(self, name: str, description: str, parameters: dict, function: Callable):
        self.name = name
        self.description = description
        self.parameters = parameters  # JSON Schema of the arguments
        self.function = function

    def run(self, arguments: dict) -> str:
        return str(self.function(**arguments))

    def __call__(self, *args, **kwargs):
        return self.function(*args, **kwargs)

In [ ]:
JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array", dict: "object"}


def tool(function: Callable) -> Tool:
    """Decorator: build a Tool from a function's name, docstring and type hints."""
    hints = get_type_hints(function, include_extras=True)
    properties, required = {}, []

    for name, param in inspect.signature(function).parameters.items():
        hint, description = hints.get(name, str), None
        if get_origin(hint) is Annotated:  # Annotated[int, "description"]
            hint, description = get_args(hint)[:2]
        properties[name] = {"type": JSON_TYPES.get(hint, "string")}
        if description:
            properties[name]["description"] = description
        if param.default is inspect.Parameter.empty:
            required.append(name)

    return Tool(
        name=function.__name__,
        description=inspect.getdoc(function) or "",
        parameters={"type": "object", "properties": properties, "required": required},
        function=function
    )

In [ ]:
@tool
def get_current_time(timezone: Annotated[str, "IANA timezone name, e.g. 'America/Toronto', 'Asia/Shanghai'"]) -> str:
    """Get the current local date and time in a given IANA timezone."""
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")


print(json.dumps(get_current_time.parameters, indent=2))
print(get_current_time("America/Toronto"))

In [ ]:
@tool
def add_days(date: Annotated[str, "Start date in YYYY-MM-DD format"],
             days: Annotated[int, "Number of days to add, negative to subtract"] = 1) -> str:
    """Add (or subtract, if negative) a number of days to a date. Returns the new date and its weekday."""
    result = datetime.strptime(date, "%Y-%m-%d") + timedelta(days=days)
    return result.strftime("%Y-%m-%d (%A)")


print(json.dumps(add_days.parameters, indent=2))
print(add_days("2026-11-10", 100))

In [ ]:
class ToolRegistry:

    def __init__(self, tools: List[Tool] = None):
        self.tools = {}
        for t in tools or []:
            self.register(t)

    def register(self, t: Tool) -> Tool:
        if t.name in self.tools:
            raise ValueError(f"duplicate tool name: '{t.name}'")
        self.tools[t.name] = t
        return t

    def list(self) -> List[Tool]:
        return list(self.tools.values())

    def run(self, call: ToolCall) -> str:
        t = self.tools.get(call.name)
        if t is None:
            return f"Error: unknown tool '{call.name}'"
        try:
            return t.run(call.arguments)
        except Exception as e:
            return f"Error: {e}"  # let the model see the error and try to fix it

In [ ]:
class MCPConnection:
    """A synchronous wrapper around the async mcp.Client (Part 6 will make the whole Agent async).

    The connection lives in one long-running coroutine on a background event loop;
    every call is submitted to that loop and we wait for its result.
    """

    def __init__(self, url: str):  # the URL of a remote MCP server
        self.loop = asyncio.new_event_loop()
        threading.Thread(target=self.loop.run_forever, daemon=True).start()
        ready = threading.Event()
        self.session = asyncio.run_coroutine_threadsafe(self._session(url, ready), self.loop)
        ready.wait()
        if self.session.done():  # connecting failed: raise the error here
            self.session.result()

    async def _session(self, url: str, ready: threading.Event):
        self.closed = asyncio.Event()
        try:
            async with Client(url) as client:
                self.client = client
                ready.set()
                await self.closed.wait()  # keep the connection open until close()
        finally:
            ready.set()

    def _run(self, coroutine):
        return asyncio.run_coroutine_threadsafe(coroutine, self.loop).result()

    def list_tools(self):
        return self._run(self.client.list_tools()).tools

    def call_tool(self, name: str, arguments: dict):
        return self._run(self.client.call_tool(name, arguments))

    def close(self):
        self.loop.call_soon_threadsafe(self.closed.set)
        self.session.result()
        self.loop.call_soon_threadsafe(self.loop.stop)

In [ ]:
class MCPTool(Tool):
    """A tool that lives on an MCP server: same interface as Tool, but run() goes over MCP."""

    def __init__(self, connection: MCPConnection, mcp_tool):
        super().__init__(mcp_tool.name, mcp_tool.description or "", mcp_tool.input_schema, function=None)
        self.connection = connection

    def run(self, arguments: dict) -> str:
        result = self.connection.call_tool(self.name, arguments)
        text = "\n".join(block.text for block in result.content if block.type == "text")
        if result.is_error and not text.startswith("Error"):
            text = f"Error: {text}"  # same convention as ToolRegistry.run()
        return text


def load_mcp_tools(connection: MCPConnection) -> List[Tool]:
    return [MCPTool(connection, t) for t in connection.list_tools()]

In [ ]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1, memory_id=None):
        self.provider = provider
        self.id = memory_id or uuid.uuid4().hex[:8]
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, *messages: Message):
        # add a whole turn at once, so it is summarized (at most) once per turn
        self.messages.extend(messages)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        # never separate a tool call from its result: keep from the last UserMessage onwards
        while split > 0 and not isinstance(self.messages[split], UserMessage):
            split -= 1
        if split == 0:
            return
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        note("summary", self.history())

    def token_count(self):
        # rough estimate: counts characters, not real tokens
        return sum(len(msg.summary() or "") for msg in self.messages)

    def to_dict(self) -> dict:
        return {"id": self.id, "messages": [msg.to_dict() for msg in self.messages]}

    @classmethod
    def from_dict(cls, data: dict, provider) -> "Memory":
        memory = cls(provider, memory_id=data["id"])
        memory.messages = [message_from_dict(m) for m in data["messages"]]
        return memory

In [ ]:
class MemoryStore:

    def __init__(self, root="memories"):
        self.root = Path(root)
        self.root.mkdir(exist_ok=True)

    def save(self, memory: Memory):
        path = self.root / f"{memory.id}.json"
        path.write_text(json.dumps(memory.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

    def load(self, memory_id: str, provider) -> Memory:
        path = self.root / f"{memory_id}.json"
        if not path.exists():
            return None
        return Memory.from_dict(json.loads(path.read_text(encoding="utf-8")), provider)

In [ ]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        pass

In [ ]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ChatCompletionToolParam(type="function", function={
                "name": tool.name, "description": tool.description, "parameters": tool.parameters})
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                param = ChatCompletionAssistantMessageParam(role="assistant", content=msg.content)
                if msg.tool_calls:
                    param["tool_calls"] = [
                        ChatCompletionMessageToolCallParam(id=call.id, type="function", function={
                            "name": call.name, "arguments": json.dumps(call.arguments)})
                        for call in msg.tool_calls
                    ]
                schema.append(param)
            elif isinstance(msg, ToolResultMessage):
                # OpenAI: one "tool" message per tool result
                schema.append(ChatCompletionToolMessageParam(role="tool", tool_call_id=msg.tool_call_id, content=msg.result))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        tool_calls = [
            ToolCall(call.id, call.function.name, json.loads(call.function.arguments))
            for call in (message.tool_calls or [])
        ]
        return AssistantMessage(message.content or "", tool_calls)

In [ ]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ToolParam(name=tool.name, description=tool.description, input_schema=tool.parameters)
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                content = [TextBlockParam(type="text", text=msg.content)] if msg.content else []
                content += [
                    ToolUseBlockParam(type="tool_use", id=call.id, name=call.name, input=call.arguments)
                    for call in msg.tool_calls
                ]
                schema.append(AssistantMessageParam(role="assistant", content=content))
            elif isinstance(msg, ToolResultMessage):
                # Claude: all tool results of one turn go into a single "user" message
                block = ToolResultBlockParam(type="tool_result", tool_use_id=msg.tool_call_id, content=msg.result)
                if schema and schema[-1]["role"] == "user" and isinstance(schema[-1]["content"], list):
                    schema[-1]["content"].append(block)
                else:
                    schema.append(UserMessageParam(role="user", content=[block]))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        tool_calls = [
            ToolCall(block.id, block.name, block.input)
            for block in response.content if block.type == "tool_use"
        ]
        return AssistantMessage(text, tool_calls)

`EmbeddingProvider` 多了一个属性 `model`，记录嵌入模型的名字。保存索引的时候会一起保存它，原因等一下讲。`OpenAIEmbedding` 在创建时已经设置了 `self.model`，所以不用改。

In [ ]:
class EmbeddingProvider(ABC):
    model: str = ""  # name of the embedding model, saved together with an index

    @abstractmethod
    def embed(self, texts: List[str]) -> np.ndarray:
        """Return one vector per text, shape (len(texts), dimension)."""
        pass

In [ ]:
class OpenAIEmbedding(EmbeddingProvider):

    def __init__(self, model: str = "text-embedding-3-small"):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.model = model

    def embed(self, texts: List[str]) -> np.ndarray:
        response = self.client.embeddings.create(model=self.model, input=texts)
        data = sorted(response.data, key=lambda d: d.index)  # one embedding per input, in order
        return np.array([d.embedding for d in data])

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Cosine similarity between every row of a and every row of b, shape (len(a), len(b))."""
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a @ b.T

In [ ]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage, registry: ToolRegistry = None, max_steps: int = 5):
        self.provider = provider
        self.system = system
        self.registry = registry or ToolRegistry()
        self.max_steps = max_steps
        self.store = MemoryStore()
        self.memory = Memory(provider)

    def run(self, message: str, system: SystemMessage = None) -> AssistantMessage:
        system = system or self.system
        tools = self.registry.list()
        turn = [UserMessage(message)]

        response = self.provider.call(system, self.memory.messages + turn, tools)
        turn.append(response)

        step = 0
        while response.tool_calls and step < self.max_steps:
            step += 1
            for call in response.tool_calls:
                result = self.registry.run(call)
                note(f"tool[{step}]", f"{call} -> {result}")
                turn.append(ToolResultMessage(call.id, call.name, result))
            response = self.provider.call(system, self.memory.messages + turn, tools)
            turn.append(response)

        if response.tool_calls:
            response.tool_calls = []
            response.content = response.content or f"(stopped after {self.max_steps} steps)"

        self.memory.add(*turn)
        self.store.save(self.memory)
        return response

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None  # preset questions, or None to type
        note("memory", f"#{self.memory.id}")
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            if message.strip().lower() == "/new":
                self.memory = Memory(self.provider)
                note("new memory", f"#{self.memory.id}")
                continue

            if message.startswith("#"):
                memory = self.store.load(message[1:].strip(), self.provider)
                if memory is None:
                    note("memory", f"{message} not found")
                else:
                    self.memory = memory
                    note(f"loaded memory #{memory.id}", memory.history())
                continue

            response = self.run(message, system)
            print(f"AI: {response.content}")

## 准备一个小知识库

我们把前面几章讲过的内容写成三篇 Markdown 文章，放在 `knowledge` 文件夹里，作为学习助手的知识库。到第 15 章，学习助手就能根据它们来回答小明的问题。

`%%writefile 文件名` 是 Jupyter 的一个“魔法命令”：写在单元格的第一行，运行时不会执行下面的内容，而是把它们原样写进文件。

In [ ]:
Path("knowledge").mkdir(exist_ok=True)  # a small knowledge base: the topics of the previous chapters

In [ ]:
%%writefile knowledge/tool-calling.md
# 工具调用

## 什么是工具调用
大模型本身只能生成文本，无法直接访问外部世界。工具调用（Tool Calling，也称 Function Calling）让模型可以在回复中请求调用某个函数：模型输出函数名和参数，由我们的程序真正执行函数，再把结果发回给模型。模型从不直接执行代码。

## 工具的定义
每个工具包含三部分：名称、描述和参数的 JSON Schema。模型主要依靠描述来决定什么时候使用哪个工具，因此描述写得是否清楚，直接影响模型的选择。

## 工具循环
一个问题可能需要多次工具调用。Agent 在一个循环中反复调用模型：只要模型还在请求工具，就执行工具并把结果发回，直到模型给出最终回答。为了防止无限循环，通常会设置最大步数。

## 错误处理
工具执行出错时，不应该让程序崩溃，而应该把错误信息作为工具结果返回给模型。模型看到错误后，往往能够修正参数并重新调用。

In [ ]:
%%writefile knowledge/mcp.md
# MCP 协议

## 概述
MCP（Model Context Protocol，模型上下文协议）是一个开放协议，用统一的方式把工具、资源和提示词提供给 AI 应用。有了 MCP，同一个工具服务器可以被不同的应用复用，而不需要为每个应用单独集成。

## 架构
MCP 采用客户端-服务器架构。宿主应用（例如一个 Agent）内部的 MCP 客户端与 MCP 服务器建立连接，服务器对外提供工具。双方使用 JSON-RPC 2.0 交换消息，连接建立时先通过 initialize 握手协商协议版本。

## 传输方式
MCP 支持两种主要的传输方式。stdio 方式下，客户端把服务器作为子进程启动，通过标准输入输出通信，适合本地工具。Streamable HTTP 方式下，服务器作为独立的网络服务运行，客户端通过 HTTP POST 发送请求，适合远程部署。早期的 HTTP+SSE 传输方式已被 Streamable HTTP 取代。

## 常见问题
使用 stdio 传输时，服务器的标准输出只能用来传输协议消息。如果在服务器代码里用 print 输出调试信息，会破坏协议数据，导致客户端报错。日志应该写到标准错误输出。

In [ ]:
%%writefile knowledge/memory.md
# 对话记忆

## 为什么需要记忆
大模型的 API 是无状态的：每次调用都是独立的，模型不会记得上一次的对话。要实现多轮对话，程序需要把历史消息保存下来，在每次调用时一起发送给模型。

## 记忆压缩
对话越长，发送的历史消息越多，成本越高，最终还会超出模型的上下文窗口。一种常见的做法是在历史过长时，让模型把较早的对话压缩成一段摘要，只保留最近的几条消息。

## 持久化
把记忆保存到文件或数据库中，Agent 重启后就能加载之前的对话继续进行。每段记忆用一个唯一的 ID 标识，用户可以随时开始新的对话，或者切换回旧的对话。

## 长期记忆
摘要压缩解决的是单次对话内的短期记忆。如果希望 Agent 记住跨越多次对话的信息，比如用户的偏好，就需要长期记忆：把重要的事实提取出来，存成向量，在需要时检索相关的记忆。

## 文档和块

`Chunk` 表示切出来的一小块，它记录两样东西：

- `text`：这一块的文字；
- `source`：它来自哪篇文章。下一章学习助手回答问题时，要用它注明出处。

`load_documents` 读取文件夹里所有的 `.md` 文件，返回文件名和内容。

In [ ]:
class Chunk:

    def __init__(self, text: str, source: str):
        self.text = text
        self.source = source  # which document it comes from, used for citations in chapter 15

    def to_dict(self) -> dict:
        return {"text": self.text, "source": self.source}


def load_documents(folder: str) -> List[tuple]:
    """Return (source, text) for every Markdown file in the folder."""
    return [(path.name, path.read_text(encoding="utf-8")) for path in sorted(Path(folder).glob("*.md"))]


documents = load_documents("knowledge")
for source, text in documents:
    print(source, len(text), "characters")

## 切块方法一：按长度切

最简单的方法是按固定长度切：每 `size` 个字切一块。

但这样很可能把一句话从中间切断，前一块的结尾和后一块的开头都变得不完整。所以通常会让相邻的两块**重叠**一部分（`overlap`），即使一句话被切断了，它在某一块里也能保持相对完整。

下面把 `mcp.md` 按每块 120 个字、重叠 30 个字来切，看看切出来的效果：

In [ ]:
def chunk_fixed(text: str, size: int = 200, overlap: int = 50) -> List[str]:
    """Cut the text every `size` characters; neighbouring chunks share `overlap` characters."""
    step = size - overlap
    return [text[start:start + size] for start in range(0, max(len(text) - overlap, 1), step)]


source, text = next(d for d in documents if d[0] == "mcp.md")
for piece in chunk_fixed(text, size=120, overlap=30):
    print(repr(piece))

可以看到，很多块都是从句子中间开始、在句子中间结束的，有的块还把两个小节的内容拼在了一起。

## 切块方法二：按标题切

我们的文章是 Markdown 格式，本身就有清楚的结构：`#` 是文章标题，`##` 是小节标题。每个小节讲的是一件事，正好可以作为一块。

`chunk_by_heading` 逐行读取文章：

- 遇到 `#` 或 `##` 标题，就把前面积累的内容作为一块保存下来（`flush`），然后记住新的标题；
- 其他的行，先积累起来。

保存每一块时，在前面加上“文章标题 > 小节标题”。比如“常见问题”这一小节，单独看不知道说的是什么，加上“MCP 协议 > 常见问题”就清楚多了，算出来的向量也更准确。

如果某个小节太长，超过了 `max_size`，就再用按长度切的方法把它切开。

In [ ]:
def chunk_by_heading(text: str, source: str, max_size: int = 500) -> List[Chunk]:
    """One chunk per "## " section, prefixed with its headings; long sections are cut again."""
    chunks, title, heading, lines = [], "", "", []

    def flush():
        body = "\n".join(lines).strip()
        if body:
            prefix = " > ".join(h for h in (title, heading) if h)
            pieces = chunk_fixed(body, max_size, overlap=50) if len(body) > max_size else [body]
            chunks.extend(Chunk(f"{prefix}\n{piece}", source) for piece in pieces)
        lines.clear()

    for line in text.splitlines():
        if line.startswith("# "):
            flush()
            title, heading = line[2:].strip(), ""
        elif line.startswith("## "):
            flush()
            heading = line[3:].strip()
        else:
            lines.append(line)
    flush()
    return chunks


for chunk in chunk_by_heading(text, source):
    print(repr(chunk.text))

> **块切多大合适？** 没有标准答案。块太大，一块里混着好几件事，向量就不够准确，交给大模型的无关内容也多；块太小，每块的信息不完整，大模型拿到了也答不全。常见的做法是几百个字一块，再根据实际效果调整。

## 向量索引 VectorIndex

`VectorIndex` 把块和它们的向量放在一起管理：

- `add`：给一批块算出向量，加进索引。嵌入接口一次能接收的文字有上限，所以每次最多发 `batch_size` 块，最后用 `np.vstack` 把各批的向量上下拼在一起。
- `search`：把问题变成向量，和所有块的向量算相似度，返回最相似的 `top_k` 块。这和上一章的 `semantic_search` 一样，只是资料的向量已经提前算好了。
- `save`：把向量保存成 `vectors.npy`（`numpy` 的文件格式），把块的文字、出处和嵌入模型的名字保存成 `chunks.json`。
- `load`：从文件夹加载索引。这里有一个检查：**索引是用哪个嵌入模型建的，查询时就必须用同一个模型**。不同模型生成的向量，就像不同语言写的地址，放在一起比较没有意义。所以模型名字对不上时，直接报错。

In [ ]:
class VectorIndex:

    def __init__(self, embedder: EmbeddingProvider):
        self.embedder = embedder
        self.chunks: List[Chunk] = []
        self.vectors = None  # shape (len(chunks), dimension)

    def add(self, chunks: List[Chunk], batch_size: int = 100):
        batches = [chunks[i:i + batch_size] for i in range(0, len(chunks), batch_size)]
        vectors = np.vstack([self.embedder.embed([c.text for c in batch]) for batch in batches])
        self.vectors = vectors if self.vectors is None else np.vstack([self.vectors, vectors])
        self.chunks += chunks

    def search(self, query: str, top_k: int = 3) -> List[tuple]:
        if not self.chunks:
            return []
        scores = cosine_similarity(self.embedder.embed([query]), self.vectors)[0]
        best = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], float(scores[i])) for i in best]

    def save(self, folder: str):
        folder = Path(folder)
        folder.mkdir(exist_ok=True)
        np.save(folder / "vectors.npy", self.vectors)
        data = {"model": self.embedder.model, "chunks": [c.to_dict() for c in self.chunks]}
        (folder / "chunks.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, folder: str, embedder: EmbeddingProvider) -> "VectorIndex":
        folder = Path(folder)
        data = json.loads((folder / "chunks.json").read_text(encoding="utf-8"))
        if data["model"] != embedder.model:  # vectors of different models can not be compared
            raise ValueError(f"index was built with '{data['model']}', not '{embedder.model}'")
        index = cls(embedder)
        index.chunks = [Chunk(**c) for c in data["chunks"]]
        index.vectors = np.load(folder / "vectors.npy")
        return index

## 建立索引

把三篇文章按标题切块，全部加进索引。

In [ ]:
embedder = OpenAIEmbedding()
index = VectorIndex(embedder)
for source, text in documents:
    index.add(chunk_by_heading(text, source))
print(len(index.chunks), "chunks, vectors:", index.vectors.shape)

试着查几个问题。`show` 函数打印每个结果的相似度、出处和第一行（也就是标题）。

In [ ]:
def show(results):
    for chunk, score in results:
        print(f"   {score:.3f}  [{chunk.source}] {chunk.text.splitlines()[0]}")


for query in ["MCP 服务器里可以用 print 调试吗？", "模型怎么知道该用哪个工具？", "Agent 重启后之前的对话还在吗？"]:
    print("Q:", query)
    show(index.search(query))

## 保存和加载

把索引保存到 `index` 文件夹，再重新加载。加载以后，文章的向量直接从文件读出来，不需要再调用嵌入接口，只有问题本身需要算一次向量。

In [ ]:
index.save("index")
loaded = VectorIndex.load("index", embedder)  # no embedding calls for the documents any more
print(len(loaded.chunks), "chunks loaded")
show(loaded.search("工具出错了怎么办？"))

> 我们的索引很简单：把所有向量放在一个数组里，每次和问题逐一比较。几千、几万块的资料完全够用。真实的项目资料可能有几百万块，就会用专门的**向量数据库**，比如 Chroma、FAISS、pgvector 等。它们用更聪明的方法快速找到最相似的向量，但原理和我们这里是一样的。

## 动手试试

- 在 `knowledge` 文件夹里加一篇你自己写的笔记，比如第 11 章的工具箱，重新建立索引，问一个相关的问题。
- 把 `chunk_by_heading` 换成 `chunk_fixed`，用同样的问题搜索，比较两种切法的结果。
- 用 `VectorIndex.load("index", OpenAIEmbedding("text-embedding-3-large"))` 加载索引，看看会发生什么。

## 小结

- 长文章要先**切块**，搜索才能精确到小段落。
- 按长度切最简单，用**重叠**减少句子被切断的影响；按标题切能保持每块意思完整，加上标题前缀让每块的内容更清楚。
- **索引**提前算好所有块的向量并保存下来，搜索时只需要计算问题的向量。
- 建索引和查询必须用**同一个嵌入模型**。

现在，学习助手已经能从知识库里找到相关的段落了。下一章，我们把找到的段落交给大模型，让学习助手真正根据这本书来回答问题，并注明出处。